In [3]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from MongoCRUD import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'GrazSalv-Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(className='row',
             style={'display': 'flex', 'align-items': 'center', 'padding': '10px'},
             children=[
                 html.A(
                     [
                         html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height': '100px'})
                     ],
                     href='https://www.snhu.edu/',
                     target='_blank'
                 ),
                 html.Div([
                     html.Center(html.B(html.H1('Austin Animal Center Dashboard'))),
                     html.Center(html.H4('Dashboard by Jacob Gilliam - 2025-08-13 - NineTwoFiveDev'))
                 ], style={'flex-grow': '1'})
             ]),
    html.Hr(),
    html.Div([
        html.H3("Filter Animals for Rescue Missions"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster Rescue or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True,
            style={'display': 'flex', 'gap': '20px', 'justify-content': 'center'}
        )
    ], style={'padding': '20px', 'text-align': 'center'}),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         column_selectable="single",
                         row_selectable="single",
                         row_deletable=False,
                         selected_columns=[],
                         selected_rows=[],
                         page_action="native",
                         page_current=0,
                         page_size=10,
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type == 'water':
        # Water rescue dogs - strong swimmers - medium to large size
        query = {
            "$and": [
                {"animal_type": "Dog"},
                {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}},
                {"sex_upon_outcome": "Intact Female"},
                {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
            ]
        }
    elif filter_type == 'mountain':
        # Mountain/wilderness rescue - strong breeds - good in rough terrain
        query = {
            "$and": [
                {"animal_type": "Dog"},
                {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}},
                {"sex_upon_outcome": "Intact Male"},
                {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
            ]
        }
    elif filter_type == 'disaster':
        # Disaster/tracking - intelligent and trainable breeds
        query = {
            "$and": [
                {"animal_type": "Dog"},
                {"breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}},
                {"sex_upon_outcome": "Intact Male"},
                {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
            ]
        }
    else:
        # Default case - return all records
        query = {}
    
    # Query the database and convert to DataFrame
    filtered_data = db.read(query)
    if filtered_data:
        df_filtered = pd.DataFrame.from_records(filtered_data)
        df_filtered.drop(columns=['_id'], inplace=True)
        return df_filtered.to_dict('records')
    else:
        # Return empty list if no matches found
        return []

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Added pie chart
    if viewData is None or len(viewData) == 0:
        # Return empty div if there's no data
        return html.Div("No data available for chart")
    
    # Convert data to DataFrame for plotting
    df_chart = pd.DataFrame.from_dict(viewData)
    
    # Count breeds and create pie chart
    breed_counts = df_chart['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    return [
        dcc.Graph(            
            figure = px.pie(breed_counts, 
                          values='count', 
                          names='breed', 
                          title='Available Animals by Breed')
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # Error handling for empty data or no selection
    if viewData is None or len(viewData) == 0:
        # Return default map with no markers when there's no data available
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Check if the dataframe is empty or if there's no row is selected
    if dff.empty or index is None or len(index) == 0:
        #  Return a map with no markers when there's no selection
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]
    
    # Because we only allow single row selection, the list can be converted to a row index here
    row = index[0]
    
    # Verify the selected row exists in the dataframe
    if row >= len(dff):
        # Return a map with no markers if the row index is out of bounds
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]
        
    # Check if all of the required columns exist and have valid data
    try:
        lat = dff.iloc[row, 13]  # latitude column
        lon = dff.iloc[row, 14]  # longitude column  
        breed = dff.iloc[row, 4]  # breed column
        name = dff.iloc[row, 9]   # name column
        
        # Verify lat/lon are valid numbers
        if pd.isna(lat) or pd.isna(lon):
            # Return map with no markers if coordinates are missing
            return [
                dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                    dl.TileLayer(id="base-layer-id")
                ])
            ]
            
    except (IndexError, KeyError):
        # Return map with no markers if column access fails
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(str(breed)),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(str(name))
                ])
            ])
        ])
    ]



app.run_server(debug=True)


Dash app running on http://127.0.0.1:13337/
